# Reproduction: the key table of "Train 400x faster Static Embedding Models" (Aarsen, HF blog, Jan 2025)

**Claim reproduced** (blog §Performance › NanoBEIR): `static-retrieval-mrl-en-v1` is **100–400× faster on CPU** than
`all-mpnet-base-v2` while reaching **87.4 %** of its NanoBEIR NDCG@10, and is **the only static model above BM25**.
Targets fixed before running: the blog's values below; ratios, not absolute q/s, because the hardware differs.
Then the recipe itself, scaled down 1/40: MS MARCO only, 1 epoch.

Runs end to end in about 10 minutes on a laptop GPU (RTX 5050). Everything here is computed by our own code in `static_emb/`
(reimplemented from scratch; 19 unit tests and the full 3-seed study in the companion repo).

In [1]:
import itertools, sys, time
import numpy as np, torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
sys.path.insert(0, ".")
from static_emb.nanobeir import NANOBEIR_DATASETS, bm25_score_fn, dense_score_fn, evaluate, load_nanobeir, ndcg_at_k
from static_emb.train import TrainConfig, train
from static_emb.static_model import StaticEmbedder
device = "cuda" if torch.cuda.is_available() else "cpu"
BLOG = {"static-retrieval-mrl-en-v1": 0.5032, "all-mpnet-base-v2": 0.5757, "potion-base-8M": 0.4421, "BM25": 0.4518}
print(device, torch.__version__)

cuda 2.14.1+cu130


## 1. NanoBEIR NDCG@10 (13 datasets × 50 queries), our evaluator

In [2]:
ours = {}
for label, name in [("static-retrieval-mrl-en-v1", "sentence-transformers/static-retrieval-mrl-en-v1"),
                    ("potion-base-8M", "minishlab/potion-base-8M"), ("all-mpnet-base-v2", "sentence-transformers/all-mpnet-base-v2")]:
    m = SentenceTransformer(name, device=device)
    bs = 128 if "mpnet" in name else 4096
    ours[label] = evaluate(dense_score_fn(lambda t: m.encode(t, batch_size=bs, normalize_embeddings=True)))["mean"]
ours["BM25"] = evaluate(bm25_score_fn())["mean"]          # bm25s + English stopwords + PyStemmer, as the blog describes
for k in BLOG:
    print(f"{k:28} blog {BLOG[k]:.4f}   ours {ours[k]:.4f}   Δ {ours[k] - BLOG[k]:+.4f}")
print(f"static / mpnet: blog {BLOG['static-retrieval-mrl-en-v1'] / BLOG['all-mpnet-base-v2']:.1%}, ours {ours['static-retrieval-mrl-en-v1'] / ours['all-mpnet-base-v2']:.1%}")

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


static-retrieval-mrl-en-v1   blog 0.5032   ours 0.5035   Δ +0.0003
all-mpnet-base-v2            blog 0.5757   ours 0.5757   Δ +0.0000
potion-base-8M               blog 0.4421   ours 0.4428   Δ +0.0007
BM25                         blog 0.4518   ours 0.5736   Δ +0.1218
static / mpnet: blog 87.4%, ours 87.5%


**BM25 does not reproduce.** Is it our BM25? A second, independent BM25 written in NumPy (Lucene IDF, k1 = 1.5, b = 0.75,
regex tokens + Porter/Snowball stems) gives the same answer:

In [3]:
import re, Stemmer
from collections import Counter
stem = Stemmer.Stemmer("english"); tok_re = re.compile(r"(?u)\b\w\w+\b")
tok = lambda s: stem.stemWords(tok_re.findall(s.lower()))
def numpy_bm25(queries, docs, k1=1.5, b=0.75):
    counts = [Counter(tok(d)) for d in docs]; vocab = {t: i for i, t in enumerate({t for c in counts for t in c})}
    tf = np.zeros((len(docs), len(vocab)), dtype=np.float32)
    for i, c in enumerate(counts):
        for t, f in c.items(): tf[i, vocab[t]] = f
    dl = tf.sum(1); df = (tf > 0).sum(0); idf = np.log(1 + (len(docs) - df + 0.5) / (df + 0.5))
    S = np.zeros((len(queries), len(docs)), dtype=np.float32)
    for qi, q in enumerate(queries):
        ids = [vocab[t] for t in set(tok(q)) if t in vocab]; f = tf[:, ids]
        S[qi] = (idf[ids] * f * (k1 + 1) / (f + k1 * (1 - b + b * dl[:, None] / dl.mean()))).sum(1)
    return S
indep = np.mean([ndcg_at_k(numpy_bm25(qs, ds), q, d, qr) for q, qs, d, ds, qr in map(load_nanobeir, NANOBEIR_DATASETS)])
print(f"independent NumPy BM25: {indep:.4f}   (bm25s: {ours['BM25']:.4f}, blog: {BLOG['BM25']:.4f})")

independent NumPy BM25: 0.5663   (bm25s: 0.5736, blog: 0.4518)


## 2. CPU speed: GooAQ queries per second (blog: 107,420 vs 270 q/s = 397×)

In [4]:
queries = [r["question"] for r in itertools.islice(load_dataset("sentence-transformers/gooaq", split="train", streaming=True), 20_000)]
def qps(model, n, bs):
    model.encode(queries[:bs], batch_size=bs); t0 = time.perf_counter(); model.encode(queries[:n], batch_size=bs)
    return n / (time.perf_counter() - t0)
static_cpu = qps(SentenceTransformer("sentence-transformers/static-retrieval-mrl-en-v1", device="cpu"), 20_000, 4096)
mpnet_cpu = qps(SentenceTransformer("sentence-transformers/all-mpnet-base-v2", device="cpu"), 1_000, 64)
print(f"static {static_cpu:,.0f} q/s · mpnet {mpnet_cpu:,.0f} q/s · ratio {static_cpu / mpnet_cpu:.0f}x on {torch.get_num_threads()} CPU threads")

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


static 68,620 q/s · mpnet 116 q/s · ratio 590x on 8 CPU threads


## 3. The recipe at 1/40 scale: train a static model from random init
Blog recipe (bert-base-uncased tokenizer, 1024 dims, MNRL + Matryoshka, batch 2048, lr 0.2, warmup 10 %, bf16, 1 epoch),
but on **one** dataset (MS MARCO triplets, 493k) instead of 13. Control: the same model before training.

In [5]:
history_eval = lambda m: {"nanobeir_mean": evaluate(dense_score_fn(lambda t: m.encode(t, normalize=True)))["mean"]}
model, hist = train(TrainConfig(seed=12, log_every=60), eval_fn=history_eval)
print(f"untrained (random table): {hist['eval'][0]['nanobeir_mean']:.4f}   trained: {hist['eval'][-1]['nanobeir_mean']:.4f}"
      f"   ({hist['n_train']:,} triplets, {hist['train_s']:.0f}s of training; blog's full recipe: 0.5032 after 17.8 h)")

step    1/241  loss 34.8937  lr 0.0080


step   60/241  loss 14.0112  lr 0.1676


step  120/241  loss 12.3958  lr 0.1120


step  180/241  loss 11.4451  lr 0.0565


step  240/241  loss 11.5815  lr 0.0009


step  241/241  loss 11.0197  lr 0.0000


untrained (random table): 0.2536   trained: 0.4628   (492,939 triplets, 10s of training; blog's full recipe: 0.5032 after 17.8 h)


## Verdict

| Claim | Blog | Ours | Verdict |
|---|---|---|---|
| NanoBEIR NDCG@10 of the static, potion and mpnet models | see above | within ±0.001 | reproduced |
| static ≈ 87 % of mpnet | 87.4 % | see above | reproduced |
| ≥ 100× faster than mpnet on CPU | 397× | see above (≈ 500–800× on this laptop) | reproduced |
| the only static model above BM25 | BM25 0.4518 | BM25 ≈ 0.57 (two implementations) | **not reproduced**: a standard BM25 beats every static model |
| the training recipe works | 0.503 (13 datasets, 17.8 h) | ≈ 0.46 (1 dataset, ~10 s) | partially, at 1/40 scale |